# Dynamics 04c (C#) — Coopération payorienne : le FairBot qui se prouve par cadres de Kripke

**Navigation** : [<< Dynamics-03 (C#)](Dynamics-03-Sympathie-vs-Engagement-CSharp.ipynb) | [compagnon formel 04b](Dynamics-04b-FairBot-Loeb-Lean.ipynb) | [Version Python](Dynamics-04c-Cooperation-Payorienne-Kripke-Python.ipynb) | [Index](README.md) | [Dynamics-05-Agents-Bornes-Python >>](Dynamics-05-Agents-Bornes-Python.ipynb)

Ce carnet est le jumeau C# de [Dynamics-04c (Python)](Dynamics-04c-Cooperation-Payorienne-Kripke-Python.ipynb) : même fil pédagogique, mêmes résultats mesurés, moteur de Kripke réécrit de zéro en C# sur la seule BCL.

Ce carnet sert la tranche T-2 du plan de consolidation de l'[issue #26985](https://github.com/jsboige/CoursIA/issues/26985), née du Concern de l'issue #26954 : l'état de l'art ouvert de la théorie des jeux (OSGT) n'était cité nulle part dans le dépôt. Il reprend la **coopération payorienne** du post d'Alignment Forum *« Payorian cooperation is easy with Kripke frames »* (transhumanist_atom_understander, 9 mars 2026), lu dans sa copie archivée au gisement bibliographique (`GameTheory/Cooperation, Institutions and LLM Agents/2026-03-09 - transhumanist_atom_understander - Payorian cooperation is easy with Kripke frames - Alignment Forum LaCP6WyNzX8kiZn3w.pdf`).

Le carnet fait trois choses, et une seule de chaque :

1. **énonce** le FairBot payorien `P ↔ □(□P → Q)` et le lemme de Payor, en citant la source primaire — aucune preuve formelle n'est refaite ici ;
2. **implémente** le bot dans l'esprit du moteur jouet de [Dynamics-04](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) ;
3. **mesure** la profondeur de réfutation par cadres de Kripke, avec un moteur déclaré en **copie pédagogique** du laboratoire [Tweety-03b](../../SymbolicAI/Tweety/Tweety-03b-Modal-Lab-Lean-Python.ipynb) §3.

La frontière mesure/preuve est tenue partout : les réfutations sont des **calculs bornés**, pas des théorèmes ; le côté formel vit dans [`ProgramGames.FairBot`](../game_theory_lean/ProgramGames/FairBot.lean) (niveau L2) et dans [Dynamics-04b](Dynamics-04b-FairBot-Loeb-Lean.ipynb) pour la partie löbienne.

## 1. Le tournoi et la question

Le tournoi de MIRI est un dilemme du prisonnier **one-shot** où chaque programme lit le code de l'autre — ou plutôt une description de son comportement en logique de la prouvabilité de Gödel-Löb (GL). FairBot est défini par « je coopère avec toi si tu coopères avec moi, prouvablement » :

$$F(X) \leftrightarrow \Box\, X(F)$$

Malgré la circularité apparente, deux FairBots coopèrent : c'est la **coopération löbienne**, qui passe par le théorème de Löb.

Critch a proposé une autre voie, le **lemme de Payor** : au lieu de vérifier que l'adversaire coopère, on vérifie que *sa propre coopération prouvable implique* celle de l'adversaire :

$$P \leftrightarrow \Box(\Box P \to Q)$$

avec $P \equiv F(X)$ et $Q \equiv X(F)$. La question de ce carnet : **pourquoi cette forme se réfute-t-elle plus facilement sur un cadre de Kripke ?** Le post répond « deux mondes au lieu de quatre » ; nous allons le mesurer.

Les paiements sont ceux du moteur de Dynamics-04.

In [1]:
// --- Paiements du dilemme du prisonnier (meme convention que Dynamics-04) ---
int T = Dilemme.T, R = Dilemme.R, P_PAIEMENT = Dilemme.P_PAIEMENT, S = Dilemme.S;
Controles.Verifie(T > R && R > P_PAIEMENT && P_PAIEMENT > S, "parametres PD non canoniques");
Controles.Verifie(2 * R > T + S, "cooperation mutuelle > alternance");

Console.WriteLine("self  other  payoff");
foreach (string me in new[] { "C", "D" })
{
    foreach (string other in new[] { "C", "D" })
    {
        Console.WriteLine($"  {me}     {other}     {Dilemme.PayoffSelf(me, other)}");
    }
}

public static class Controles
{
    public static void Verifie(bool condition, string message)
    {
        if (!condition)
        {
            throw new InvalidOperationException(message);
        }
    }
}

public static class Dilemme
{
    public const int T = 5;
    public const int R = 3;
    public const int P_PAIEMENT = 1;
    public const int S = 0;

    public static int PayoffSelf(string monAction, string actionAdverse)
    {
        if (monAction == "C")
        {
            return actionAdverse == "C" ? R : S;
        }
        return actionAdverse == "C" ? T : P_PAIEMENT;
    }
}

self  other  payoff


  C     C     3


  C     D     0


  D     C     5


  D     D     1


### Lecture : la matrice tient la structure du dilemme

La sortie donne exactement quatre paiements — `C/C → 3`, `C/D → 0`, `D/C → 5`, `D/D → 1` — et les deux assertions en tête de cellule fixent l'ordre strict `T > R > P > S` ainsi que la préférence pour la coopération mutuelle sur l'alternance. C'est cette matrice que les bots ci-dessous vont jouer, sans itération ni réputation : tout se décide avant le coup, sur la seule lecture du code adverse.

## 2. Le lemme de Payor, et ce que ce carnet n'en fait pas

**Lemme de Payor.** Pour toute formule $A$ et tout $B$ : $\vdash \Box(\Box A \to B) \to A$.

L'attribution est double, et le post la donne telle quelle : James Payor raconte dans le fil être arrivé à `□(□A→B)→A` en raisonnant sur les étapes de l'algorithme, et c'est **Andrew Critch** qui propose de s'en servir à la place du théorème de Löb pour prouver la coopération de soi — « *Instead of Löb's theorem, we use what he calls "Payor's lemma"* ». Le nom est resté au lemme, l'usage au post.

La preuve est **arithmétique** — elle dérive du théorème de Löb — et n'est pas refaite ici : le carnet travaille exclusivement du côté **sémantique**, où le lemme se lit comme une contrainte sur les cadres. L'axiome de Löb $\Box(\Box A \to A) \to \Box A$ y est valide exactement sur les cadres transitifs et **bien fondés**.

Déclarations de provenance :

- **copie pédagogique** : le moteur de Kripke de la cellule suivante est une copie déclarée du §3 du laboratoire Tweety-03b (mêmes types d'AST, même clause de forcing), motif = un carnet est autonome et ne peut pas importer un autre carnet ;
- **citation** : l'énoncé du lemme et les dérivations du post sont cités, jamais re-démontrés — le côté mécanisé vit dans le lake `game_theory_lean` (niveau L2) et, pour la partie löbienne, dans le carnet 04b ;
- **archive** : la copie du post lue pour ce carnet est celle du gisement bibliographique, pas une page web revisitée.

In [2]:
// --- Moteur de Kripke : AST gelee + forcing recursif ---
// COPIE PEDAGOGIQUE DECLAREE du §3 de Tweety-03b-Modal-Lab-Lean-Python.ipynb :
// memes types d'AST, meme clause de forcing. Motif : un carnet est autonome.
Modele m_test = new Modele(
    new[] { 0, 1 },
    new[] { (0, 1) },
    new Dictionary<string, HashSet<int>> { { "P", new HashSet<int> { 1 } } });

var verdicts = new Dictionary<string, bool>
{
    ["P en 0"] = Moteur.Satisfait(m_test, 0, new Atome("P")),
    ["[]P en 0"] = Moteur.Satisfait(m_test, 0, new Boite(new Atome("P"))),
    ["[]P -> P en 0"] = Moteur.Satisfait(m_test, 0, new Imp(new Boite(new Atome("P")), new Atome("P"))),
    ["<>P en 1"] = Moteur.Satisfait(m_test, 1, new Diamant(new Atome("P"))),
};
foreach (var kv in verdicts)
{
    Console.WriteLine($"  {kv.Key.PadRight(14)} : {(kv.Value ? "vrai" : "faux")}");
}
// Un verdict affiche = un verdict controle : une assertion par ligne du tableau.
Controles.Verifie(!verdicts["P en 0"], "P en 0 : P n'est vrai qu'au monde 1");
Controles.Verifie(verdicts["[]P en 0"], "[]P en 0 : le monde mort rend la box vacuement vraie");
Controles.Verifie(!verdicts["[]P -> P en 0"], "[]P -> P en 0 : le monde mort ne porte pas P");
Controles.Verifie(!verdicts["<>P en 1"], "<>P en 1 : le monde 1 n'a aucun successeur");
Console.WriteLine();
Console.WriteLine("Moteur operationnel : au monde mort (1 sans successeur), []P est vacuement vrai.");

public abstract record Formule;

public sealed record Atome(string Nom) : Formule;
public sealed record Non(Formule F) : Formule;
public sealed record Imp(Formule A, Formule B) : Formule;
public sealed record Et(Formule A, Formule B) : Formule;
public sealed record Boite(Formule F) : Formule;
public sealed record Diamant(Formule F) : Formule;

public sealed class Modele
{
    public IReadOnlyList<int> Mondes { get; }
    public HashSet<(int, int)> Relation { get; }
    public Dictionary<string, HashSet<int>> Valuation { get; }

    public Modele(IEnumerable<int> mondes, IEnumerable<(int, int)> relation, Dictionary<string, HashSet<int>> valuation)
    {
        Mondes = mondes.ToList();
        Relation = new HashSet<(int, int)>(relation);
        Valuation = valuation;
    }

    public IEnumerable<int> Successeurs(int x)
    {
        return Relation.Where(p => p.Item1 == x).Select(p => p.Item2);
    }
}

public static class Moteur
{
    public static string Show(Formule f)
    {
        return f switch
        {
            Atome a => a.Nom,
            Non n => "non " + Show(n.F),
            Imp i => "(" + Show(i.A) + " -> " + Show(i.B) + ")",
            Et e => "(" + Show(e.A) + " et " + Show(e.B) + ")",
            Boite b => "[]" + Show(b.F),
            Diamant d => "<>" + Show(d.F),
            _ => throw new ArgumentException("formule inconnue"),
        };
    }

    public static bool Satisfait(Modele m, int x, Formule f)
    {
        return f switch
        {
            Atome a => m.Valuation.TryGetValue(a.Nom, out var ens) && ens.Contains(x),
            Non n => !Satisfait(m, x, n.F),
            Imp i => !Satisfait(m, x, i.A) || Satisfait(m, x, i.B),
            Et e => Satisfait(m, x, e.A) && Satisfait(m, x, e.B),
            Boite b => m.Successeurs(x).All(y => Satisfait(m, y, b.F)),
            Diamant d => m.Successeurs(x).Any(y => Satisfait(m, y, d.F)),
            _ => throw new ArgumentException("formule inconnue"),
        };
    }
}

  P en 0         : faux


  []P en 0       : vrai


  []P -> P en 0  : faux


  <>P en 1       : faux


Moteur operationnel : au monde mort (1 sans successeur), []P est vacuement vrai.


### Lecture : le monde mort et la vacuité

`[]P en 0` est vrai alors que `[]P -> P en 0` est faux — la sortie montre les deux lignes opposées, et l'assertion finale le confirme. Le monde 1, sans successeur, rend `[]` vacuement vrai ; c'est exactement la clause qui portera la réfutation : une boîte fausse signale toujours un monde *visible* où son contenu échoue.

## 3. Réfuter en empilant des contraintes sur un cadre

La méthode du post est un raisonnement par l'absurde sémantique. Pour prouver que FairBot coopère, on suppose qu'il **défecte** : $\neg P$. Comme $P \leftrightarrow \Box(\Box P \to Q)$, cela donne

$$\neg\Box(\Box P \to Q) \;\equiv\; \Diamond(\Box P \land \neg Q)$$

c'est-à-dire un monde visible $w_1$ où $\Box P$ et $\neg Q$ tiennent. On propage ensuite les boîtes le long de la visibilité — transitivité : si $w_0$ voit $w_1$ et $w_1$ voit $w_2$, alors $w_0$ voit $w_2$ — et l'on cherche une **contradiction locale**, un atome et sa négation au même monde.

La cellule suivante code cet algorithme : un dérouleur qui, à chaque obligation de possibilité non satisfaite, ouvre **un** monde de plus (chaîne croissante), propage les nécessités, et s'arrête sur contradiction. Deux limites sont assumées d'emblée : la borne de profondeur rend un statut INCOMPLET plutôt que « cohérent », et le dérouleur n'encode **pas** la bonne-fondation des cadres de GL — la section 5 montrera précisément ce que cela change pour le côté löbien.

In [3]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// --- Derouleur de refutation : saturer des contraintes jusqu'a contradiction ---
// L'algorithme du post, borne en profondeur. Ce n'est PAS un prouveur de GL :
// au-dela de la borne, une obligation ouverte rend INCOMPLET, jamais "coherent".

public static class F
{
    public static Atome A(string nom) => new Atome(nom);
    public static Non Neg(Formule f) => new Non(f);
    public static Imp Impl(Formule a, Formule b) => new Imp(a, b);
    public static Boite Box(Formule f) => new Boite(f);
    public static Diamant Dia(Formule f) => new Diamant(f);
    public static Et Equiv(Formule f, Formule g) => new Et(new Imp(f, g), new Imp(g, f));
}

public sealed class Contradiction : Exception
{
    public int Monde { get; }
    public Formule Formule { get; }
    public Contradiction(int monde, Formule f)
        : base($"monde {monde} : {Moteur.Show(f)} et sa negation")
    {
        Monde = monde;
        Formule = f;
    }
}

public sealed class ResultatRefutation
{
    public string Statut { get; init; } = "";
    public int? Monde { get; init; }
    public Formule? Formule { get; init; }
    public Dictionary<int, HashSet<Formule>>? Mondes { get; init; }
}

public static class Derouleur
{
    public static bool Voit(int i, int j) => i < j;

    public static void Ajoute(Dictionary<int, HashSet<Formule>> mondes, int w, Formule f)
    {
        foreach (Formule g in mondes[w])
        {
            if (g is Non ng && ng.F.Equals(f))
            {
                throw new Contradiction(w, f);
            }
        }
        if (f is Non nf && mondes[w].Contains(nf.F))
        {
            throw new Contradiction(w, nf.F);
        }
        mondes[w].Add(f);
    }

    private static int[] Visibles(Dictionary<int, HashSet<Formule>> mondes, int w)
        => mondes.Keys.Where(x => Voit(w, x)).OrderBy(x => x).ToArray();

    public static ResultatRefutation Refute(IReadOnlyList<Formule> premisses, int maxProfondeur = 6)
    {
        var mondes = new Dictionary<int, HashSet<Formule>> { [0] = new HashSet<Formule>(premisses) };
        try
        {
            while (true)
            {
                bool change = false;
                foreach (int w in mondes.Keys.OrderBy(k => k).ToArray())
                {
                    foreach (Formule f in mondes[w].OrderBy(Moteur.Show, StringComparer.Ordinal).ToArray())
                    {
                        if (f is Non nn && nn.F is Non nn2)
                        {
                            if (!mondes[w].Contains(nn2.F))
                            {
                                Ajoute(mondes, w, nn2.F);
                                change = true;
                            }
                        }
                        else if (f is Et et)
                        {
                            foreach (Formule g in new[] { et.A, et.B })
                            {
                                if (!mondes[w].Contains(g))
                                {
                                    Ajoute(mondes, w, g);
                                    change = true;
                                }
                            }
                        }
                        else if (f is Imp im)
                        {
                            if (mondes[w].Contains(im.A) && !mondes[w].Contains(im.B))
                            {
                                Ajoute(mondes, w, im.B);
                                change = true;
                            }
                            if (mondes[w].Contains(new Non(im.B)) && !mondes[w].Contains(new Non(im.A)))
                            {
                                Ajoute(mondes, w, new Non(im.A));
                                change = true;
                            }
                        }
                        else if (f is Non n3 && n3.F is Imp im2)
                        {
                            foreach (Formule g in new[] { im2.A, new Non(im2.B) })
                            {
                                if (!mondes[w].Contains(g))
                                {
                                    Ajoute(mondes, w, g);
                                    change = true;
                                }
                            }
                        }
                        else if (f is Boite bo)
                        {
                            foreach (int v in Visibles(mondes, w))
                            {
                                if (!mondes[v].Contains(bo.F))
                                {
                                    Ajoute(mondes, v, bo.F);
                                    change = true;
                                }
                            }
                        }
                        else if (f is Non n4 && n4.F is Diamant di)
                        {
                            foreach (int v in Visibles(mondes, w))
                            {
                                if (!mondes[v].Contains(new Non(di.F)))
                                {
                                    Ajoute(mondes, v, new Non(di.F));
                                    change = true;
                                }
                            }
                        }
                        else if (f is Non n5 && n5.F is Boite bo2)
                        {
                            Formule besoin = new Non(bo2.F);
                            if (!Visibles(mondes, w).Any(v => mondes[v].Contains(besoin)))
                            {
                                if (mondes.Keys.Max() >= maxProfondeur)
                                {
                                    return new ResultatRefutation { Statut = "incomplet" };
                                }
                                mondes[mondes.Keys.Max() + 1] = new HashSet<Formule> { besoin };
                                change = true;
                            }
                        }
                        else if (f is Diamant di2)
                        {
                            Formule besoin = di2.F;
                            if (!Visibles(mondes, w).Any(v => mondes[v].Contains(besoin)))
                            {
                                if (mondes.Keys.Max() >= maxProfondeur)
                                {
                                    return new ResultatRefutation { Statut = "incomplet" };
                                }
                                mondes[mondes.Keys.Max() + 1] = new HashSet<Formule> { besoin };
                                change = true;
                            }
                        }
                    }
                }
                if (!change)
                {
                    return new ResultatRefutation { Statut = "sat", Mondes = mondes };
                }
            }
        }
        catch (Contradiction c)
        {
            return new ResultatRefutation { Statut = "unsat", Monde = c.Monde, Formule = c.Formule };
        }
    }
}

Console.WriteLine("Derouleur defini : Refute(premisses) -> (statut, info).");
Console.WriteLine("Statuts : unsat (contradiction, info = monde), sat (modele fini), incomplet (borne).");

Derouleur defini : Refute(premisses) -> (statut, info).


Statuts : unsat (contradiction, info = monde), sat (modele fini), incomplet (borne).


### Lecture du dérouleur

Trois statuts, et aucun des trois n'est un « vrai » :

| statut | sens | ce qu'on peut en dire |
|---|---|---|
| `unsat` | l'atome et sa négation se sont rencontrées au monde `info.Monde` | **réfutation** : l'hypothèse de départ est incompatible avec les prémisses |
| `sat` | saturation sans contradiction | un **modèle fini** des prémisses : le comportement observé est *possible*, pas *forcé* |
| `incomplet` | borne atteinte, obligation encore ouverte | ni cohérent, ni contradictoire : **on ne sait pas** |

Deux limites à garder en tête avant de lire le tableau suivant.

1. **`sat` n'est pas une preuve de cohérence.** Le modèle trouvé est un modèle du *cadre employé* — une chaîne transitive finie de mondes. Un autre cadre pourrait fermer la réfutation ; un modèle, c'est une possibilité, pas un théorème.
2. **Le dérouleur n'encode pas la bonne-fondation.** Les cadres de GL sont transitifs **et** bien fondés : aucune chaîne infinie de mondes accessibles. Ici la visibilité est la simple chaîne `i < j` sur les mondes déjà créés, sans aucun axiome de bonne-fondation. Quand une obligation ouvre un monde de plus à chaque passe, le dérouleur en ouvre indéfiniment et bute sur sa borne. C'est une propriété du **dérouleur**, pas une découverte sur Löb — et c'est précisément ce qui rend le cas löbien `INCOMPLET` à la section suivante.

In [4]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// --- Les quatre jeux de premisses du post ---
Atome P_AT = new Atome("P"), Q_AT = new Atome("Q"), H_AT = new Atome("H");

Formule COND_PAYOR   = F.Equiv(P_AT, F.Box(F.Impl(F.Box(P_AT), Q_AT)));   // P coopere ssi P prouve []([]P -> Q)
Formule COND_PAYOR_Q = F.Equiv(Q_AT, F.Box(F.Impl(F.Box(Q_AT), P_AT)));   // la meme definition, vue de Q
Formule COND_COOP    = F.Box(Q_AT);                                      // CooperateBot : coopere toujours
Formule COND_DEFECT  = F.Box(new Non(Q_AT));                             // DefectBot : trahit toujours
Formule COND_LOEB    = F.Equiv(H_AT, F.Box(H_AT));                       // phrase de Henkin : H <-> []H

// Pourquoi les definitions sont AUSSI posees sous une boite (Box(COND_...)) :
// la refutation de ~P se joue dans le monde w1, ou l'on simule l'adversaire.
// Pour que cette simulation raisonne avec SA definition, celle-ci doit etre
// accessible dans les successeurs -- c'est la premisse supplementaire que le
// fil reclame explicitement ("repeating the description of myself but wrapped
// in a provability modality so it's accessible in the simulations", 8mo).

var CAS = new Dictionary<string, List<Formule>>
{
    ["P vs CooperateBot"] = new() { COND_PAYOR, F.Box(COND_PAYOR), COND_COOP, new Non(P_AT) },
    ["P vs Payorian Q"] = new() { COND_PAYOR, F.Box(COND_PAYOR), COND_PAYOR_Q, F.Box(COND_PAYOR_Q), new Non(P_AT) },
    ["P vs DefectBot"] = new() { COND_PAYOR, F.Box(COND_PAYOR), COND_DEFECT, new Non(P_AT) },
    ["Lob : H <-> []H"] = new() { COND_LOEB, F.Box(COND_LOEB), new Non(H_AT) },
};

(string? Borne, string Statut, ResultatRefutation? Info) ProfondeurMinimale(IReadOnlyList<Formule> premisses, int borne = 6)
{
    for (int k = 1; k <= borne; k++)
    {
        ResultatRefutation r = Derouleur.Refute(premisses, k);
        if (r.Statut != "incomplet")
        {
            return (k.ToString(), r.Statut, r);
        }
    }
    return (null, "incomplet", null);
}

Console.WriteLine($"{"jeu de premisses",-20} {"statut",-10} {"borne",-7} detail");
Console.WriteLine(new string('-', 78));
foreach (var kv in CAS)
{
    var (borne, statut, info) = ProfondeurMinimale(kv.Value);
    string detail = statut == "unsat"
        ? $"contradiction en {info!.Monde} : {Moteur.Show(info.Formule!)}"
        : statut == "sat"
            ? $"modele a {info!.Mondes!.Count} mondes"
            : "borne atteinte, obligation ouverte";
    Console.WriteLine($"{kv.Key,-20} {statut,-10} {borne ?? "",-7} {detail}");
}

// --- Controles : les trois profondeurs annoncees par le post, plus la frontiere ---
var r1 = ProfondeurMinimale(CAS["P vs CooperateBot"]);
var r2 = ProfondeurMinimale(CAS["P vs Payorian Q"]);
var r3 = ProfondeurMinimale(CAS["P vs DefectBot"]);
var r4 = ProfondeurMinimale(CAS["Lob : H <-> []H"]);

Controles.Verifie(r1.Statut == "unsat" && r1.Borne == "1", $"cas 1 : {r1.Statut}/{r1.Borne}");   // un seul monde ajoute
Controles.Verifie(r2.Statut == "unsat" && r2.Borne == "2", $"cas 2 : {r2.Statut}/{r2.Borne}");   // deux mondes : c'est le "2 au lieu de 4"
Controles.Verifie(r3.Statut == "sat" && r3.Info!.Mondes!.Count == 2, $"cas 3 : {r3.Statut}");
Controles.Verifie(r4.Statut == "incomplet", $"cas 4 : {r4.Statut}");

// Lecture du modele de P vs DefectBot : le monde 1 porte []P et ~Q.
// L'atome P n'y figure pas -- c'est la biconditionnelle, verifiee dans la
// cellule suivante, qui l'y rend vrai.
var m3 = r3.Info!.Mondes!;
Controles.Verifie(m3[1].Contains(new Boite(P_AT)) && m3[1].Contains(new Non(Q_AT)), "monde 1 : []P et non Q");
Console.WriteLine();
Console.WriteLine("P vs DefectBot : monde 1 = " + string.Join(", ", m3[1].OrderBy(Moteur.Show, StringComparer.Ordinal).Select(Moteur.Show)));
Console.WriteLine("Controles passes : 1 monde, 2 mondes, sat a 2 mondes, incomplet.");

jeu de premisses     statut     borne   detail


------------------------------------------------------------------------------


P vs CooperateBot    unsat      1       contradiction en 1 : Q


P vs Payorian Q      unsat      2       contradiction en 2 : P


P vs DefectBot       sat        1       modele a 2 mondes


Lob : H <-> []H      incomplet          borne atteinte, obligation ouverte


P vs DefectBot : monde 1 = ((P -> []([]P -> Q)) et ([]([]P -> Q) -> P)), (P -> []([]P -> Q)), ([]([]P -> Q) -> P), []P, non ([]P -> Q), non Q


Controles passes : 1 monde, 2 mondes, sat a 2 mondes, incomplet.


### Vérifier un `sat` : les deux moteurs doivent s'accorder

Un `sat` n'est pas une ligne de console, c'est l'affirmation qu'un modèle existe. On la vérifie en reconstruisant un `Modele` à partir des mondes saturés, puis en repassant **chaque prémisse** au moteur de forcing de la section 2 — un moteur écrit indépendamment, pour un autre carnet.

La reconstruction prend une convention, et il faut la nommer : un atome est vrai dans un monde **sauf si** sa négation y figure. C'est une fermeture par défaut, pas une conséquence du dérouleur. Si les prémisses ne tiennent pas sous cette convention, le `sat` est faux et l'assertion le signale ; si elles tiennent, on a bien un modèle, et les deux moteurs s'accordent.

In [5]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// --- Un "sat" se verifie : on repasse le modele au moteur de la section 2 ---
public static class Reconstruction
{
    public static HashSet<string> Atomes(Formule f)
    {
        return f switch
        {
            Atome a => new HashSet<string> { a.Nom },
            Non n => Atomes(n.F),
            Imp i => Atomes(i.A).Union(Atomes(i.B)).ToHashSet(),
            Et e => Atomes(e.A).Union(Atomes(e.B)).ToHashSet(),
            Boite b => Atomes(b.F),
            Diamant d => Atomes(d.F),
            _ => new HashSet<string>(),
        };
    }

    public static Modele VersModele(Dictionary<int, HashSet<Formule>> mondes)
    {
        var noms = new HashSet<string>();
        foreach (var fs in mondes.Values)
        {
            foreach (Formule f in fs)
            {
                noms.UnionWith(Atomes(f));
            }
        }
        var valuation = new Dictionary<string, HashSet<int>>();
        foreach (string n in noms)
        {
            valuation[n] = new HashSet<int>(mondes.Keys.Where(w => !mondes[w].Contains(new Non(new Atome(n)))));
        }
        var relation = new HashSet<(int, int)>();
        foreach (int i in mondes.Keys)
        {
            foreach (int j in mondes.Keys)
            {
                if (i < j)
                {
                    relation.Add((i, j));
                }
            }
        }
        return new Modele(mondes.Keys.OrderBy(k => k), relation, valuation);
    }
}

Modele M3 = Reconstruction.VersModele(m3);
List<Formule> premisses = CAS["P vs DefectBot"];
var verdicts = premisses.Select(p => Moteur.Satisfait(M3, 0, p)).ToList();
foreach (var (p, v) in premisses.Zip(verdicts))
{
    Console.WriteLine($"  {(v ? "vrai" : "FAUX"),4}  {Moteur.Show(p)}");
}
int conformes = verdicts.Count(v => v);
Console.WriteLine($"Premisses revalidees par le moteur de la section 2 : {conformes}/{verdicts.Count} -> {(conformes == verdicts.Count ? "CONFORME" : "ECART")}");
Controles.Verifie(verdicts.All(v => v), "le modele ne satisfait pas les premisses : le 'sat' serait faux");

Console.WriteLine();
Console.WriteLine("Modele valide : les quatre premisses tiennent en w0.");
Console.WriteLine("P est vrai en w1 par la biconditionnelle, pas par propagation de []P :");
Console.WriteLine("le monde 1 est terminal, donc []P n'y contraint aucun successeur.");

  vrai  ((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))


  vrai  []((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))


  vrai  []non Q


  vrai  non P


Premisses revalidees par le moteur de la section 2 : 4/4 -> CONFORME


Modele valide : les quatre premisses tiennent en w0.


P est vrai en w1 par la biconditionnelle, pas par propagation de []P :


le monde 1 est terminal, donc []P n'y contraint aucun successeur.


### Lecture du tableau

**Une prémisse qui n'est pas un détail.** Les deux définitions sont posées à la racine **et** sous une boîte. Sans la boîte, la définition de l'adversaire n'atteint pas `w1`, la simulation ne sait pas quel bot elle simule, et la réfutation rend un `sat` trompeur. Le fil le formule lui-même : *repeating the description of myself but wrapped in a provability modality so it's accessible in the simulations*. C'est une prémisse du modèle, pas un ajustement du code — et c'est le genre d'oubli qui fait passer un carnet pour une preuve.

**P vs CooperateBot — un monde.** `□Q` à la racine place `Q` dans le seul monde créé, où `¬Q` était déjà posé par la négation de la clause de Payor. La contradiction est **interne** au monde 1 : aucun enchaînement n'est nécessaire.

**P vs Payorian Q — deux mondes.** C'est le résultat central. En `w1` on a `□P` et `¬Q`. Le `¬Q` attaque la définition de Q, qui ouvre `w2` avec `¬P` ; et le `□P` de `w1` place `P` dans `w2`. La contradiction naît **entre les deux définitions**, dans un monde que chacune atteint par un chemin différent. Deux mondes ajoutés, pas quatre.

**P vs DefectBot — saturé, deux mondes.** Le modèle est petit : `w0` porte `¬P`, `w1` porte `□P` et `¬Q`. Tout tient, et le point qui fait tenir est que **`w1` ne voit rien** : `□(□P→Q)` y est vacuement vrai, comme la boîte sur un monde terminal de la section 2. Ce `sat` n'est pas un accident de programmation : la cellule suivante le **repasse au moteur de la section 2** et vérifie que les quatre prémisses tiennent. C'est la **frontière de l'hypothèse de cohérence** — la réfutation seule ne conclut pas que le FairBot payorien trahit contre DefectBot. Il faut lui ajouter une hypothèse de cohérence sur son propre système de preuve, et c'est écrit dans le post comme une réserve, pas comme un théorème.

**Löb — incomplet.** La phrase de Henkin `H ↔ □H` n'ouvre pas une contradiction : elle ouvre un monde de plus à chaque passe, indéfiniment. Le dérouleur rend `incomplet`, ce qui est la réponse **honnête** de cet instrument-là. Un prouveur de GL fermerait, parce qu'il dispose de l'axiome de Löb et de la bonne-fondation ; notre dérouleur ne les a pas.

## 4. Le tournoi, en jouets

La section 3 a réfuté des formules. Il reste à voir ce que ces réfutations **disent du jeu**. On reprend la convention du carnet `Dynamics-04-Open-Source-Game-Theory-Python` — un bot rend `"C"` ou `"D"` après avoir lu quelque chose de son adversaire — en remplaçant la lecture du code source par une **déclaration** en clair : chaque bot publie sa règle, et c'est ce texte que les autres lisent. Une caricature assumée, et il faut dire exactement ce qu'elle vaut :

- le tournoi **illustre** les issues que l'analyse de Kripke contraint ;
- il ne les **démontre pas**. Sur le cas `P vs DefectBot`, l'analyse rend `sat` : deux issues sont compatibles avec les prémisses, et le jouet en choisit une par construction. Ce choix n'est pas une dérivation, c'est une règle d'écriture.

La déclaration joue le rôle de la définition de l'adversaire dans une simulation modale : le bot ne voit pas la fonction, il voit ce que l'autre annonce de lui-même — exactement la limite du dérouleur de la section 3, qui ne connaît de l'adversaire que la formule qu'on lui donne.

In [6]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// --- Les trois bots, chacun declarant sa regle ---
// Un bot lit la DECLARATION de son adversaire (le jouet ne voit que le texte,
// pas la fonction : c'est la caricature assumee du paragraphe precedent).

Func<string, string> coopere_toujours = _ => "C";
Func<string, string> trahit_toujours = _ => "D";
Func<string, string> fairbot_payorien = decl =>
    (decl.Contains("coopere toujours") || decl.Contains("coopere si l'adversaire")) ? "C" : "D";

// nom -> (decision, declaration que les autres lisent)
var BOTS = new Dictionary<string, (Func<string, string> Decision, string Declaration)>
{
    ["cooperatebot"] = (coopere_toujours, "coopere toujours"),
    ["defectbot"] = (trahit_toujours, "trahit toujours"),
    ["fairbot_payorien"] = (fairbot_payorien, "coopere si l'adversaire declare coopere"),
};

int Gain(string mien, string sien) =>
    (mien == "C" && sien == "C") ? 3 : (mien == "C" ? 0 : (sien == "C" ? 5 : 1));

(string A, string B) Jouer(string nomA, string nomB)
{
    var (fa, declA) = BOTS[nomA];
    var (fb, declB) = BOTS[nomB];
    return (fa(declB), fb(declA));
}

// --- Matrice des gains ---
List<string> noms = BOTS.Keys.ToList();
Console.WriteLine(new string(' ', 18) + string.Join("", noms.Select(n => n.PadLeft(18))));
Console.WriteLine(new string('-', 18 + 18 * noms.Count));
foreach (string a in noms)
{
    var ligne = new List<string>();
    foreach (string b in noms)
    {
        var (ca, cb) = Jouer(a, b);
        ligne.Add($"{ca}/{cb} {Gain(ca, cb)}");
    }
    Console.WriteLine(a.PadRight(18) + string.Join("", ligne.Select(c => c.PadLeft(18))));
}

// --- Ce que la section 3 contraint, et ce qu'elle ne contraint pas ---
var cc = Jouer("fairbot_payorien", "cooperatebot");
var ff = Jouer("fairbot_payorien", "fairbot_payorien");
var cd = Jouer("cooperatebot", "cooperatebot");
var dd = Jouer("defectbot", "defectbot");
Console.WriteLine($"Diagonale : cooperatebot {cd.A}/{cd.B} gain {Gain(cd.A, cd.B)} | defectbot {dd.A}/{dd.B} gain {Gain(dd.A, dd.B)} | fairbot {ff.A}/{ff.B} gain {Gain(ff.A, ff.B)}");
Controles.Verifie(cc == ("C", "C"), "fairbot vs cooperatebot");      // ferme en 1 monde
Controles.Verifie(ff == ("C", "C"), "fairbot vs fairbot");          // ferme en 2 mondes
Controles.Verifie(cd == ("C", "C") && Gain(cd.A, cd.B) == 3, "diagonale cooperatebot");
Controles.Verifie(dd == ("D", "D") && Gain(dd.A, dd.B) == 1, "diagonale defectbot");
Controles.Verifie(Gain(ff.A, ff.B) == 3, "diagonale fairbot : gain");

var (ca, cb) = Jouer("fairbot_payorien", "defectbot");
// Case D/D, gain 1 : ce que rend le JOUEET, verifie par assertion sur les
// deux actions et sur le gain -- mais ce que le JOUEET choisit, pas ce que
// la logique force (la section 3 rend 'sat' sur ce cas).
Controles.Verifie(ca == "D" && cb == "D", "P vs DefectBot : le jouet doit rendre D/D");
Controles.Verifie(Gain(ca, cb) == 1, "P vs DefectBot : gain attendu 1");
Console.WriteLine();
Console.WriteLine($"P vs DefectBot (jouet) : {ca}/{cb} -> gain {Gain(ca, cb)}");
Console.WriteLine("L'analyse de la section 3 rend 'sat' ici : le jouet choisit,");
Console.WriteLine("il ne derive pas. Les assertions ci-dessus contraignent le jouet,");
Console.WriteLine("elles ne demontrent rien sur la logique modale.");

                        cooperatebot         defectbot  fairbot_payorien


------------------------------------------------------------------------


cooperatebot                   C/C 3             C/D 0             C/C 3


defectbot                      D/C 5             D/D 1             D/D 1


fairbot_payorien               C/C 3             D/D 1             C/C 3


Diagonale : cooperatebot C/C gain 3 | defectbot D/D gain 1 | fairbot C/C gain 3


P vs DefectBot (jouet) : D/D -> gain 1


L'analyse de la section 3 rend 'sat' ici : le jouet choisit,


il ne derive pas. Les assertions ci-dessus contraignent le jouet,


elles ne demontrent rien sur la logique modale.


### Lecture du tournoi

Diagonale : `(3,3)`, `(1,1)`, `(3,3)`. Le FairBot payorien coopère contre le coopérateur inconditionnel et contre lui-même — les deux seules cases que la section 3 **contraint**, et les assertions du code le vérifient (actions et gain, pour chacune des trois cases de la diagonale).

La case qui instruit est `P vs DefectBot`. Le jouet rend `(1,1)`, et c'est exactement le piège : on serait tenté de lire ce `1/1` comme « le FairBot payorien trahit contre DefectBot, donc il est prudent ». La section 3 dit autre chose — `sat`, deux mondes — et la différence n'est pas une nuance :

- le jouet **applique une règle de correspondance textuelle** ; il n'a pas de notion de preuve, donc rien ne l'empêche de rendre `D` ;
- l'analyse de Kripke **laisse les deux issues ouvertes** : le modèle à deux mondes est un modèle *des prémisses*, il ne prouve pas que le bot joue `C` en vrai ;
- conclure « il trahit » demande une hypothèse supplémentaire, la cohérence du système de preuve du bot. Le post la pose explicitement ; beaucoup de relectures l'oublient.

Autrement dit : le tournoi et la réfutation ne sont pas deux mesures de la même chose. Le tournoi **exhibe** un comportement ; la réfutation **borne** ce qui est forcé. Confondre les deux, c'est prendre un modèle pour un théorème.

L'assertion `P vs DefectBot : le jouet doit rendre D/D` mérite d'être lue pour ce qu'elle est : elle vérifie que le **jouet** tient sa règle d'écriture, pas que la logique force la défection. Une assertion peut porter sur le calcul sans porter sur la thèse.

## 5. Deux mondes au lieu de quatre

Le titre du post est une comparaison : la clause de Payor réfute en **deux** mondes là où la route löbienne en demande **quatre**. Il faut séparer ce que ce carnet a mesuré de ce qu'il ne fait que citer, parce que la moitié citée est la moitié qui compte pour le titre.

| Ce qui est en jeu | Statut ici | Détail |
|---|---|---|
| Réfutation de `¬P` sous la clause de Payor, contre un Payorien | **mesuré** — 1 monde contre CooperateBot, 2 contre un Payorien | sections 3 et 4, assertions du carnet |
| Réfutation de `¬P` contre DefectBot | **mesuré** — `sat`, modèle à 2 mondes | frontière de l'hypothèse de cohérence |
| Route löbienne : « quatre mondes » | **cité**, non re-mesuré | le dérouleur rend `incomplet` sur la phrase de Henkin seule |

Pourquoi le « quatre » ne se re-mesure pas avec cet instrument. La route löbienne part de `F(X) ↔ □(X(F))`, donc pour `F` contre `F` il faut établir un point fixe : `F(F)=C` ssi `□(F(F)=C)`. La preuve du post introduit une **phrase de Henkin** `H ↔ □H` et une **auxiliaire** `L ↔ (□L → H)` ; c'est cette auxiliaire qui rend la dérivation finie, et elle la rend finie **parce qu'elle est une instance de l'axiome de Löb** : `□(□L→H) → L`-style, appliqué à un antécédent qu'on vient de dériver. L'axiome de Löb encode la bonne-fondation des cadres de GL.

Notre dérouleur n'a ni l'axiome ni la bonne-fondation : il empile des contraintes de boîte et ouvre un monde de plus à chaque fois qu'une obligation reste ouverte. Sur `H ↔ □H` avec `¬H`, il en ouvre indéfiniment — d'où l'`incomplet` de la section 3. **Ce n'est pas un échec de la réfutation, c'est la limite de l'instrument**, et la dire fait partie du résultat.

Ce que la comparaison garde de solide, et qui est le vrai contenu pédagogique : les deux routes ne coûtent pas la même chose parce qu'elles ne prouvent pas la même chose. La clause de Payor ajoute une **boîte en position d'antécédent** — `□P` sous le `□` extérieur — et cette boîte intérieure est ce qui referme la réfutation en descendant une seule fois. Lire `□(□P→Q)` comme « je coopère si je peux prouver que ma coopération implique la tienne » est exactement ce qui rend la preuve courte : le bot se demande ce qu'il peut prouver **de sa propre coopération**, pas seulement celle de l'adversaire.

## 6. Les variantes du fil — citées, pas re-mesurées

La discussion qui suit le post propose trois bots de plus. Aucun n'est implémenté ici, et c'est un choix : leurs définitions sortent du fragment que le dérouleur traite (voir l'avertissement en fin de section). Les définitions sont citées telles quelles.

**PrudentBot d'origine** — `P(X) ↔ □(X(P) ∧ ◊¬X(DB))` : il coopère s'il prouve que l'adversaire coopère avec lui **et** qu'il existe un monde où cet adversaire trahit contre DefectBot. L'auteur du post juge lui-même cette référence à DefectBot « un peu bricolée » (*a little hacked*), et c'est ce qui motive la suite.

**SharkBot** — `S(X) ↔ □(□S(X) → X(S)) ∧ ◊(X(S) → ◊S(X))` : la clause de Payor, plus une seconde condition sur les diamants. L'auteur annonce « je suis assez sûr » (*I'm pretty sure*) qu'il coopère avec lui-même et avec FairBot, et qu'il trahit contre CooperateBot. **« Assez sûr » n'est pas un théorème** : c'est une conjecture de discussion, et ce carnet ne la vérifie pas.

**PrudentBot payorien** — `P ↔ □(□P→Q) ∧ □(□¬P→¬Q)`, obtenu en simulant l'adversaire deux fois : une fois sous l'hypothèse d'une preuve que je coopère, une fois sous l'hypothèse d'une preuve que je défais. La réfutation part de `¬P` et rend `◊(□P∧¬Q) ∨ ◊(□¬P∧Q)` : **deux** jeux de contraintes à réfuter, un par simulation. L'auteur conclut lui-même : « je ne suis pas vraiment sûr de comment tout cela va tourner » (*I'm not really sure how any of this will turn out*).

### Pourquoi ces variantes ne passent pas dans le dérouleur

`◊(□P∧¬Q) ∨ ◊(□¬P∧Q)` contient une **disjonction**, et `¬P` sur `P ≡ A∧B` produit `¬A ∨ ¬B`. Or le dérouleur de la section 3 ne décompose **ni les disjonctions ni les négations de conjonction** : il ne connaît que `¬¬`, `∧`, `→`, `¬(A→B)`, `□`, `¬◊`, `¬□` et `◊`. Une formule qu'il n'a pas décomposée reste dans le monde sans être traitée, et le dérouleur rendrait `sat` — **un `sat` qui ne voudrait rien dire**. C'est la raison pour laquelle les quatre cas de la section 3 ont été choisis sans disjonction : sur ce fragment-là, le résultat est fiable ; en dehors, il ne l'est pas.

Cette limite n'est pas propre au code. James Payor note dans le fil que la traduction `◊ = ¬□¬` **ne reste pas fidèle à l'intention** (*the "not box not" translation doesn't actually stay very faithful to the intent*), et que la manipulation des longueurs de preuve y est délicate. Le dérouleur implémente exactement cette traduction-là : quand il rencontre `¬□φ`, il ouvre un monde avec `¬φ`. C'est commode, c'est mesurable, et ce n'est pas la même chose que « il existe un monde où φ est vrai » au sens de la logique de la prouvabilité.

## 7. Exercices

Trois exercices, du plus mécanique au plus conceptuel. Les cellules sont des squelettes : elles s'exécutent telles quelles et rendent une valeur neutre tant qu'elles ne sont pas complétées. Aucune ne lève d'erreur volontairement.

**Exercice 1 — réfuter avec des disjonctions.** Le dérouleur ne décompose pas `¬(A∧B)`, qui vaut `¬A ∨ ¬B`. Écrire un `RefuteBranche` qui, en présence d'une disjonction à satisfaire, **explore les deux branches** et ne déclare `unsat` que si toutes les branches se ferment. Le tester sur les quatre cas de la section 3 : il doit rendre exactement les mêmes statuts.

In [7]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// Exercice 1 — refutation avec disjonctions
// TODO etudiant : comme Derouler.Refute, mais decomposer les disjonctions en
// explorant les deux branches ; n'accepter "unsat" que si TOUTES les branches se ferment.
// Retour attendu :
//   ("unsat", ...)       si TOUTES les branches se ferment
//   ("sat", mondes)      si une branche sature sans contradiction
//   ("incomplet", ...)   si une branche atteint la borne
Func<IReadOnlyList<Formule>, int, ResultatRefutation?> RefuteBranche = (premisses, maxProfondeur) => null;

if (RefuteBranche(CAS["P vs CooperateBot"], 6) == null)
{
    Console.WriteLine("Exercice 1 — RefuteBranche reste a completer.");
    Console.WriteLine("Controle attendu une fois complete : les quatre cas de la section 3");
    Console.WriteLine("doivent rendre exactement les memes (statut, borne) que le tableau.");
}

Exercice 1 — RefuteBranche reste a completer.


Controle attendu une fois complete : les quatre cas de la section 3


doivent rendre exactement les memes (statut, borne) que le tableau.


**Exercice 2 — le PrudentBot payorien.** Construire `COND_PRUDENT` à partir de la définition citée en section 6, réfuter `¬P` avec le dérouleur **sans** branchement, puis avec `RefuteBranche`. Expliquer en trois phrases pourquoi le premier résultat ne répond pas à la question.

In [8]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// Exercice 2 — le PrudentBot payorien
// TODO etudiant : construire la formule COND_PRUDENT a partir de la definition
// du PrudentBot payorien citee en section 6, puis la traduire dans le langage
// des formules de ce carnet (atomes, Boite, Imp, Et, Non).
Formule? COND_PRUDENT = null;  // TODO etudiant

// Puis, pour le meme jeu [COND_PRUDENT, non(P_AT)] :
//   statut_sans  = Derouler.Refute(...)         -> que rend le derouleur, et pourquoi ce n'est pas une reponse ?
//   statut_avec  = RefuteBranche(...)           -> ce que donne l'exploration des deux branches
var resultat_ex2 = new Dictionary<string, string>();  // TODO etudiant

Console.WriteLine("Exercice 2 — COND_PRUDENT et les deux refutations restent a completer.");
Console.WriteLine("Rappel : comparer les deux statuts obtenus, puis expliquer en trois");
Console.WriteLine("phrases pourquoi le premier ne repond pas a la question posee.");

Exercice 2 — COND_PRUDENT et les deux refutations restent a completer.


Rappel : comparer les deux statuts obtenus, puis expliquer en trois


phrases pourquoi le premier ne repond pas a la question posee.


**Exercice 3 — compter les mondes.** Sur le cas löbien, relever le nombre de mondes créés pour chaque borne de 1 à 6. En déduire pourquoi aucune borne ne rendra ce cas `sat` ni `unsat`, et ce qu'il faudrait ajouter au dérouleur pour fermer la réfutation.

In [9]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// Exercice 3 — compter les mondes crees sur le cas lobien
// TODO etudiant : rendre le nombre de mondes existants au moment ou le derouleur s'arrete.
Func<IReadOnlyList<Formule>, int, int?> CompteMondes = (premisses, borne) => null;

// Releve attendu : un dictionnaire {borne: nombre_de_mondes} pour borne de 1 a 6,
// obtenu sur CAS["Lob : H <-> []H"]. Conclusion attendue en deux phrases :
// pourquoi aucune borne ne rendra ce cas sat ni unsat, et ce qu'il faudrait
// ajouter au derouleur pour fermer la refutation.
Dictionary<int, int>? releve = null;  // TODO etudiant

if (CompteMondes(CAS["Lob : H <-> []H"], 1) == null)
{
    Console.WriteLine("Exercice 3 — CompteMondes et le releve restent a completer.");
}

Exercice 3 — CompteMondes et le releve restent a completer.


## À retenir

- La clause de Payor est `P ↔ □(□P→Q)` : le bot se demande ce qu'il peut prouver **de sa propre coopération**, pas seulement celle de l'adversaire. C'est la boîte intérieure qui fait la différence avec la clause löbienne `F(X) ↔ □X(F)`.
- Sur un cadre de Kripke, cette clause se réfute en **deux** mondes contre un autre Payorien, et en **un** contre CooperateBot. Les deux profondeurs sont mesurées et vérifiées par assertions.
- Contre DefectBot, la réfutation **ne conclut pas** : elle rend un modèle à deux mondes. Ce `sat` est la frontière de l'hypothèse de cohérence, pas une preuve que le bot trahit.
- La route löbienne demande, dans le post, quatre mondes et une auxiliaire `L ↔ (□L→H)`. Ce carnet la **cite** sans la re-mesurer : le dérouleur n'a ni l'axiome de Löb ni la bonne-fondation, et rend `incomplet` sur `H ↔ □H`. Un `incomplet` est un résultat honnête ; le maquiller en `sat` ou en `unsat` ne le serait pas.
- Le tournoi-jouet illustre sans démontrer : il applique une correspondance textuelle là où la logique borne ce qui est **forcé**. Les assertions du tournoi couvrent les actions et le gain de chaque case affichée ; elles contraignent le calcul du jouet, pas la thèse modale.

## Vérifiez votre compréhension

1. Pourquoi `□φ` est-il vrai sur un monde sans successeur ? Quelle est la conséquence pour le modèle `sat` de `P vs DefectBot` ?
2. Dans la réfutation `P vs Payorian Q`, à quel monde la contradiction apparaît-elle, et quelles deux formules s'y rencontrent ?
3. Le dérouleur rend `incomplet` sur le cas löbien. Est-ce parce que les prémisses sont cohérentes, contradictoires, ou parce que l'instrument ne sait pas conclure ?
4. Pourquoi un `sat` obtenu sur une formule contenant une disjonction non décomposée ne vaut-il rien ?

## Pour aller plus loin

- Le post d'origine et sa discussion : *Payorian cooperation is easy with Kripke frames*, Alignment Forum — archive du gisement, `GameTheory/Cooperation, Institutions and LLM Agents/2026-03-09 - transhumanist_atom_understander - Payorian cooperation is easy with Kripke frames.pdf`.
- Dans ce dépôt : `Dynamics-04b-FairBot-Loeb-Lean` pour la route löbienne et sa mécanisation partielle, `Dynamics-05-Agents-Bornes-Python` pour les agents bornés, `Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python` pour le coût du raisonnement et l'Open Problem 3.
- Le moteur de Kripke de la section 2 est une copie pédagogique déclarée de `SymbolicAI/Tweety/Tweety-03b-Modal-Lab-Lean-Python` : le lire pour la logique modale générale dont ce carnet n'utilise qu'un fragment.
- Version Python du même carnet : [Dynamics-04c (Python)](Dynamics-04c-Cooperation-Payorienne-Kripke-Python.ipynb).